# Overview

This assignment is designed to help you better understand TF-IDF and BM25 algorithms through implementing them from scratch, and applying them on the large-scale community Question-Answering (cQA) [LinkSO](https://dl.acm.org/doi/10.1145/3283812.3283815) dataset. Upon the successful completion of this assignment, you will develop a clear picture on how TF-IDF and BM25 algorithms work.

In this assignment, you will need to **run through** (i.e., running each code cell sequentially or `Runtime -> Run all`) the following pipeline:

- Step 0: load the dataset into the current environment;
- Step 1: import the libraries and helper functions;
- Step 2: preprocess the dataset to get the required inputs for the TF-IDF and BM25 algorithms;
- Step 3: implement the TF-IDF and BM25 algorithms.

We have already implemented Steps 1 through 3 of the pipeline. Step 4 is left for you to implement; it has three questions worthy of 30, 30, and 40 points, respectively.

**Suggestion**: Even though the code for Steps 1 through 3 has been provided, reading through all the code cells is **strongly recommended** before you start coding. Once you understand the provided code, the number of lines of code is unlikely to exceed 30.

**Runtime**: If you implement everything correctly, you will finish running this entire notebook within 90 minutes under `Stevens-Net`. If you are experiencing noticeable delay, check your network condition.

**Library requirement**: You are **NOT** allowed to use existing libraries that have already provided TF-IDF and BM25 implementations; other than these libraries, you are free to choose your preferred libraries.

**Note on Colab**: If a cell contains only a function, you should still run it (so the function compiles in Colab), even though it doesn't print any outputs.



## Step 0 - Load Dataset


Load and unzip the dataset from our remote GitHub repository that hosts the data and starter code.




In [2]:
# Remove existing cs589 directories (Windows compatible)
import shutil
import os
if os.path.exists('cs589assignment1'):
    shutil.rmtree('cs589assignment1')

In [3]:

# Clone repository and extract dataset (Windows compatible)
import subprocess
import zipfile
import os

# Clone the repository
subprocess.run(['git', 'clone', 'https://github.com/guanqun-yang/cs589assignment1.git'], check=True)

# Extract zip files
dataset_path = 'cs589assignment1/dataset'
for file in os.listdir(dataset_path):
    if file.endswith('.zip'):
        with zipfile.ZipFile(os.path.join(dataset_path, file), 'r') as zip_ref:
            zip_ref.extractall(dataset_path)


## Step 1 - Import Libraries


Import the required libraries here. You could use additional libraries to help with your implementation.

In [4]:
import re
import os
import copy
import math
import random
import string
import pathlib
import itertools

import numpy as np
import pandas as pd

from tqdm import tqdm
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import CountVectorizer

from cs589assignment1.utils.common import save_pickle_file, load_pickle_file, load_text_file

base_path = pathlib.Path("cs589assignment1/dataset/")
tqdm.pandas()

In [5]:
def split_text(text):
    return text.split()


def load_qids(lang="java"):
    return [qid.strip(string.whitespace) for qid in load_text_file(base_path / pathlib.Path(f"{lang}/{lang}_test_qid.txt"))]


def load_qid_dataframe(lang="java"):
    qid_dataframe = pd.read_csv(base_path / pathlib.Path(f"{lang}/{lang}_cosidf.txt"),
                                sep="\t",
                                usecols=["qid1", "qid2", "label"],
                                dtype={"qid1": str, "qid2": str, "label": int})
    return qid_dataframe


def load_corpus(lang="java", verbose=False):
    lines = load_text_file(base_path / pathlib.Path(f"{lang}/{lang}_qid2all.txt"))

    record_list = list()
    for line in tqdm(lines, disable=not verbose):
        record_list.append(
            {name: text.strip(string.whitespace) for name, text in zip(["qid", "title", "question", "answer"], line.split("\t"))}
        )

    corpus_dataframe = pd.DataFrame(record_list)

    return corpus_dataframe

In [6]:
# take a look at the corpus
pd.set_option("display.max_columns", 10)


java_corpus_dataframe = load_corpus(lang="python", verbose=True)
print(java_corpus_dataframe.head())

100%|██████████| 128500/128500 [00:00<00:00, 272396.91it/s]


        qid                                              title  \
0  44214957     full size page background image ignore margins   
1   3145746      running pyflakes remotely flymake tramp emacs   
2  23068700                                embedding python qt   
3  35651620  slice x coordinates shapely polygon typeerror ...   
4  43341148           training model adding new entities spacy   

                                            question  \
0  trying create pdf via python library html conv...   
1  trying use flymake run pyflakes suggested work...   
2  would like embed python interpreter qt applica...   
3  slice x coordinates shapely polygon getting fo...   
4  trying train model method using test case ques...   

                                              answer  
0  really see problem want background image cover...  
1  need tell flymake copy buffer locally prefer u...  
2  offending line problem line slots keyword defa...  
3  need retrieve exterior interior linear ring

## Step 2 - Data Preprocessing



The following cell computes the term frequency (TF) for each word in each component in each StackOverflow question (indexed by the question ID `qid`).

In [7]:
def get_corpus_tf_dict(corpus_dataframe):
    """ Input: corpus_dataframe, e.g.,

         qid         title                 question          answer
 0  31424546   eclipse mars   eclipse moving eclipse    jdk download

        Output: corpus_tf_dict, the term frequency for each word in each component of each question, e.g.,
        {'31424546': {'title': {'eclipse': 1, 'mars': 1},
                      'question': {'moving': 1, 'eclipse': 2},
                      'answer': {'jdk': 1, 'download': 1}}}
    """
    cnt_dataframe = copy.deepcopy(corpus_dataframe)
    for c in ["title", "question", "answer"]:
        cnt_dataframe[c] = cnt_dataframe[c].progress_apply(lambda x: Counter(split_text(x)))

    corpus_tf_dict = cnt_dataframe.set_index("qid").to_dict("index")

    return corpus_tf_dict



The following cell computes the document length (dl) of each component in each StackOverflow question (indexed by the question ID `qid`).


In [8]:
def get_corpus_dl_dict(corpus_dataframe):
    """ Input: corpus_dataframe, e.g.,
         qid         title                 question          answer
0  31424546   eclipse mars   eclipse moving eclipse    jdk download

        Output: corpus_dl_dict, the document length for each component from each question, e.g.,
        {'31424546': {'title': 2,
                      'question': 3,
                      'answer': 2}}
    """
    length_dataframe = copy.deepcopy(corpus_dataframe)
    for c in ["title", "question", "answer"]:
        length_dataframe[c] = length_dataframe[c].progress_apply(lambda x: len(split_text(x)))

    corpus_dl_dict = length_dataframe.set_index("qid").to_dict("index")

    return corpus_dl_dict

The following cell computes the document frequency (DF) of each word in each StackOverflow question (indexed by the question ID `qid`). The definition of document frequency is how many document a word appears in, not to be confused with the word's frequency in the entire corpus. For example, the df of "eclipse" below is 2 instead of 3.

In [9]:
def get_corpus_df_dict(corpus_dataframe):
    """ Input: corpus_dataframe, e.g.,
         qid          title                 question          answer
 0  31424546   eclipse mars   eclipse moving eclipse    jdk download

        Output: corpus_df_dict, the document length for each component from each question, e.g.,
        {'eclipse': 2, "mars": 1, "moving": 1, "jdk": 1, "download": 1}
    """
    vectorizer = CountVectorizer(binary=True)

    X = vectorizer.fit_transform(corpus_dataframe.title.tolist() + \
                                 corpus_dataframe.question.tolist() + \
                                 corpus_dataframe.answer.tolist())
    corpus_df_dict = {token: doc_freq for token, doc_freq in \
                      zip(vectorizer.get_feature_names_out(), np.ravel(X.sum(axis=0)))}

    return corpus_df_dict

### Saving the Data Preprocessing Result

After computing the TF, DF and dl, cache each of them in a pickle file to be loaded later:

In [10]:
pkl_path = pathlib.Path("pkl/")
if not pkl_path.exists(): pkl_path.mkdir()

def save_preprocessing_results(lang):
    print(f"Processing {lang}...")

    lang_pkl_path = pkl_path / lang
    if not lang_pkl_path.exists(): os.mkdir(lang_pkl_path)

    # load corpus and convert corpus to various required data
    corpus_dataframe = load_corpus(lang=lang, verbose=True)

    # obtain the dictionary for the term frequency for each word in each component of each question
    corpus_tf_dict = get_corpus_tf_dict(corpus_dataframe)

    # saving the term frequency dictionary
    save_pickle_file(corpus_tf_dict, f"pkl/{lang}/corpus_tf_dict.pkl")

    # obtain the dictionary for the document length for each component in each question
    corpus_dl_dict = get_corpus_dl_dict(corpus_dataframe)

    # save the document length dictionary
    save_pickle_file(corpus_dl_dict, f"pkl/{lang}/corpus_dl_dict.pkl")

    # obtain the dictionary for the document frequency for each word in the corpus
    corpus_df_dict = get_corpus_df_dict(corpus_dataframe)

    # remove rare words
    corpus_df_dict = {k: v for k, v in corpus_df_dict.items() if v >= 20}

    # save the document frequency dictionary
    save_pickle_file(corpus_df_dict, f"pkl/{lang}/corpus_df_dict.pkl")

    return corpus_tf_dict, corpus_dl_dict, corpus_df_dict

Run the data processing pipeline for the 3 languages:

In [11]:
for lang in ["python", "java", "javascript"]:
     save_preprocessing_results(lang)

Processing python...


100%|██████████| 128500/128500 [00:00<00:00, 207058.17it/s]


Processing java...


100%|██████████| 159263/159263 [00:00<00:00, 189650.40it/s]


Processing javascript...


100%|██████████| 174015/174015 [00:00<00:00, 190682.01it/s]


Create the folder `result` to store results for Question 4 - 6 if it does not exist.

In [12]:
result_path = pathlib.Path("result")
if not result_path.exists(): result_path.mkdir()

## Step 3 - Implement the TF-IDF and BM25 Algorithms






### Question 1 (30 pts)
Compute the cosine similarity given dictionaries of word count, `query_dict` and `candidate_dict`.
When working with term frequencies, they are extracted from `corpus_tf_dict` using `corpus_tf_dict[qid][component]` syntax).

In [13]:
def compute_cosine_similarity(query_word_cnt_dict,
                              candidate_word_cnt_dict):
    """ Input: query_dict: a dict of word and its word count in query document, e.g.
               {"i": 1, "love": 1, "python": 1}
               candidate_dict: a dict of word and its word count in the candidate document, e.g.
               {"i": 1, "like": 1, "c++": 1}
        Output: score: cosine similary between query and candidate documents
                0.33333333333333337

    """

    score = 0
    #############################################START HERE#############################################
    # Question 1 (30 pts)
    # Get the set of all words in either query or candidate
    all_words = set(query_word_cnt_dict.keys()).union(candidate_word_cnt_dict.keys())

    # Compute dot product
    dot_product = 0
    for word in all_words:
        dot_product += query_word_cnt_dict.get(word, 0) * candidate_word_cnt_dict.get(word, 0)

    # Compute norms
    query_norm = sum(v ** 2 for v in query_word_cnt_dict.values()) ** 0.5
    candidate_norm = sum(v ** 2 for v in candidate_word_cnt_dict.values()) ** 0.5

    # Avoid division by zero
    if query_norm == 0 or candidate_norm == 0:
        score = 0
    else:
        score = dot_product / (query_norm * candidate_norm)
    ##############################################END HERE##############################################
    return score

Test your `compute_cosine_similarity` implementation on the Python corpus when retrieving candidate's title using query's title.

In [14]:
lang = "python"

corpus_tf_dict = load_pickle_file(f"pkl/{lang}/corpus_tf_dict.pkl")
qid_dataframe = load_qid_dataframe(f"{lang}")

result_dict = dict()
for qid1, qid2 in list(qid_dataframe[["qid1", "qid2"]].to_records(index=False)):
    result_dict[(qid1, qid2)] = compute_cosine_similarity(corpus_tf_dict[qid1]["title"],
                                                          corpus_tf_dict[qid2]["title"])


result_filename = pathlib.Path("result/Q4.txt")
if result_filename.exists(): os.remove(result_filename)

with open(result_filename, "a") as fp:
    fp.write("qid1\tqid2\tscore\n")
    for (qid1, qid2), score in result_dict.items():
        fp.write(f"{qid1}\t{qid2}\t{score}\n")

### Question 2 (30 pts)
Compute the TF-IDF score of each word in `document_tf_dict` and store it in the `document_word_tfidf_dict`.

For the total number of documents `N`, as our LinkSO dataset is scraped from the StackOverflow website, it is a small sample of the entire pool of posts, and the exact number of posts is constantly changing (see real-time statistics [here](https://api.stackexchange.com/docs/info#filter=default&site=stackoverflow&run=true) for all topics). For the sake of this assignment, we could set the total number of posts to a constant, for example, `N = 10 ** 6`, as an approximation.

Notice the example provided as docstring is used to help you understand the input and output data structures. You are not expected to reproduce the numbers exactly.

In [15]:
def compute_document_tfidf(document_tf_dict,
                           vocab):
    """ Input: document_tf_dict: a dict of word and its term frequency in document
               {"i": 1, "love": 1, "python": 1}
               corpus_df_dict: a dict of word and its document frequencey in the entire corpus
               {"i": 2, "you": 1, "we": 3, "love": 1, "like": 1, "hate": 2, "python": 5, "c++": 3}
        Output: document_word_tfidf_dict: a dict of word and its TF-IDF score in the document
               {'i': 13.592366256649782, 'love': 14.103192380416024, 'python': 12.803907396283263}
    """

    document_word_tfidf_dict = dict()
    #############################################START HERE#############################################
    # Question 2 (30 pts)
    
    N = 10 ** 6  # Total number of documents as specified
    document_word_tfidf_dict = {
        word: (tf * math.log(N / (1 + vocab.get(word, 1))))
        for word, tf in document_tf_dict.items()
    }

    ##############################################END HERE##############################################

    return document_word_tfidf_dict

Test your `compute_document_tfidf` implementation on the title component of the Java corpus.

In [16]:
lang = "java"

corpus_tf_dict = load_pickle_file(f"pkl/{lang}/corpus_tf_dict.pkl")
corpus_df_dict = load_pickle_file(f"pkl/{lang}/corpus_df_dict.pkl")
qid_dataframe = load_qid_dataframe(f"{lang}")

result_dict = dict()
for qid1 in qid_dataframe.qid1.tolist():
    result_dict[qid1] = compute_document_tfidf(corpus_tf_dict[qid1]["title"],
                                               corpus_df_dict)

result_filename = pathlib.Path("result/Q5.txt")
if result_filename.exists(): os.remove(result_filename)

with open(result_filename, "a") as fp:
    fp.write("qid1\ttoken\ttfidf\n")
    for qid1, d in result_dict.items():
        for token, score in d.items():
            fp.write(f"{qid1}\t{token}\t{score}\n")

### Question 3 (40 pts)

Compute the BM25 score between `query_word_cnt_dict` and `candidate_word_cnt_dict`. `N = 10 ** 6` following Question 2.

Notice the example provided as docstring is used to help you understand the input and output data structures. You are not expected to reproduce the numbers exactly.

In [17]:
def compute_document_bm25(query_word_cnt_dict,
                          candidate_word_cnt_dict,
                          corpus_df_dict,
                          candidate_length,
                          avgdl,
                          k1 = 3, 
                          b = 0.75):
    """ Input: query_tf_dict: a dict of word and its term frequency in query document
               {"i": 1, "love": 1, "python": 1}
               candidate_tf_dict:a dict of word and its term frequency in candidate document
               {"i": 1, "like": 1, "c++": 1}
               corpus_df_dict: a dict of word and its document frequencey in the entire corpus
               {"i": 2, "you": 1, "we": 3, "love": 1, "like": 1, "hate": 2, "python": 5, "c++": 3}
               candidate_length: number of words in candidate document
               3
               avgdl: average document length in the entire corpus
               4
       Output: score: BM25 score between query and candidate
               15.816571644101565
    """


    # hyperparameters for BM25 algorithm shifted inside funstion parameter

    score = 0
    #############################################REPAIR THIS WRONG SOL#############################################
    # Question 3 (40 pts)
    for word, tf in query_word_cnt_dict.items():
        df = corpus_df_dict.get(word, 0)
        idf = math.log((10 ** 6 - df + 0.5) / (df + 0.5) + 1)
        tf_d = candidate_word_cnt_dict.get(word, 0)     #BM25’s term frequency must come from the candidate/document, not the query. 
                                                        #The loop can stay over query terms, 
                                                        #but the tf used in the BM25 fraction must be candidate_word_cnt_dict.get(word, 0)
        numerator = tf_d * (k1 + 1)
        denominator = tf_d + k1 * (1 - b + b * (candidate_length / avgdl))
        score += idf * (numerator / denominator)

    ##############################################END HERE##############################################

    return score


Test your `compute_document_bm25` implementation on the `title` component of the JavaScript corpus.

In [18]:
lang = "javascript"

corpus_tf_dict = load_pickle_file(f"pkl/{lang}/corpus_tf_dict.pkl")
corpus_df_dict = load_pickle_file(f"pkl/{lang}/corpus_df_dict.pkl")
corpus_dl_dict = load_pickle_file(f"pkl/{lang}/corpus_dl_dict.pkl")

qid_dataframe = load_qid_dataframe(f"{lang}")

corpus_dataframe = load_corpus(lang=lang, verbose=True)
avgdl = corpus_dataframe["title"].apply(lambda x: len(split_text(x))).sum() / len(corpus_dataframe)

result_dict = dict()
for qid1, qid2 in list(qid_dataframe[["qid1", "qid2"]].to_records(index=False)):
    result_dict[(qid1, qid2)] = compute_document_bm25(corpus_tf_dict[qid1]["title"],
                                                      corpus_tf_dict[qid2]["title"],
                                                      corpus_df_dict,
                                                      corpus_dl_dict[qid2]["title"],
                                                      avgdl)


result_filename = pathlib.Path("result/Q6.txt")
if result_filename.exists(): os.remove(result_filename)

with open(result_filename, "a") as fp:
    fp.write("qid1\tqid2\tscore\n")
    for (qid1, qid2), score in result_dict.items():
        fp.write(f"{qid1}\t{qid2}\t{score}\n")

100%|██████████| 174015/174015 [00:00<00:00, 290910.41it/s]


### Running Your Ranking Algorithms

The function `run_retrieval_algorithm` puts your implementations (`compute_cosine_similarity`, `compute_document_tfidf`, and `compute_document_bm25`) together and apply them to the entire dataset. Even though the code has been provided, it is recommended to read it to get a sense of how the retrieval pipeline works.

In [36]:
base_path = pathlib.Path("cs589assignment1/dataset/")

def run_retrieval_algorithm(lang, algo, component, qid1s=None , k1=3.0, b=0.75):
    corpus_tf_dict = load_pickle_file(f"pkl/{lang}/corpus_tf_dict.pkl")
    corpus_dl_dict = load_pickle_file(f"pkl/{lang}/corpus_dl_dict.pkl")
    corpus_df_dict = load_pickle_file(f"pkl/{lang}/corpus_df_dict.pkl")

    corpus_dataframe = load_corpus(lang=lang, verbose=False)
    available_ids = corpus_dataframe.qid.unique()
    avgdl = corpus_dataframe[component].apply(lambda x: len(split_text(x))).sum() / len(corpus_dataframe)

    qid1s = qid1s if qid1s != None else load_qids(lang=lang)

    qid1_dataframe = load_qid_dataframe(lang=lang)

    result_folder = pathlib.Path("result/")
    if not result_folder.exists(): result_folder.mkdir()

    result_filename = pathlib.Path(f"result/{lang}_{algo}_{component}_k1{float(k1):.2f}_b{float(b):.2f}.txt")

    # remove existing result file
    if result_filename.exists():
        os.remove(result_filename)

    # write header
    with open(result_filename, "a") as fp:
        fp.write("qid1\tqid2\tscore\tlabel\n")

    for qid1 in tqdm(qid1s):
        if qid1 not in available_ids: continue

        cond1 = qid1_dataframe.qid1 == qid1
        cond2 = qid1_dataframe.label == 1

        qid2s = qid1_dataframe[cond1].qid2.tolist()
        qid2s_linked = qid1_dataframe[cond1 & cond2].qid2.tolist()

        qid1_tf_dict = corpus_tf_dict[qid1]["title"]
        query_result = dict()

        # only for BM25
        max_bm25 = -1
        for qid2 in qid2s:
            if qid2 not in available_ids: continue

            qid2_tf_dict = corpus_tf_dict[qid2][component]

            # tfidf
            if algo == "tfidf":
                score = compute_cosine_similarity(compute_document_tfidf(qid1_tf_dict, corpus_df_dict),
                                                  compute_document_tfidf(qid2_tf_dict, corpus_df_dict))

            # bm25
            if algo == "bm25":
                candidate_length = corpus_dl_dict[qid2][component]
                score = compute_document_bm25(qid1_tf_dict,
                                              qid2_tf_dict,
                                              corpus_df_dict,
                                              candidate_length,
                                              avgdl,
                                              k1=k1, b=b)

                max_bm25 = max(score, max_bm25)

            query_result[qid2] = score

        # adjust BM25 score
        if (algo == "bm25") and (max_bm25 != 0):
            query_result = {qid: score / max_bm25 for qid, score in query_result.items()}

        qid2s_sorted = sorted(query_result, key=query_result.get, reverse=True)

        with open(result_filename, "a") as fp:
            for qid2 in qid2s_sorted:
                label = 1 if qid2 in qid2s_linked else 0
                score = query_result[qid2]

                fp.write(f"{qid1}\t{qid2}\t{score}\t{label}\n")

Run the retrieval algorithms and save the ranking results for each language and each retrieval algorithms:

In [20]:
langs = ["python", "java", "javascript"]
algos = ["bm25", "tfidf"]
components = ["title", "question", "answer"]

for lang, algo, component in itertools.product(langs, algos, components):
    print(f"Running {algo} on {lang}'s {component}...")
    run_retrieval_algorithm(lang, algo, component)

Running bm25 on python's title...


100%|██████████| 1000/1000 [08:13<00:00,  2.03it/s]


Running bm25 on python's question...


100%|██████████| 1000/1000 [06:13<00:00,  2.68it/s]


Running bm25 on python's answer...


...

100%|██████████| 1000/1000 [07:53<00:00,  2.11it/s]


Running tfidf on javascript's question...


100%|██████████| 1000/1000 [08:13<00:00,  2.03it/s]


Running tfidf on javascript's answer...


100%|██████████| 1000/1000 [07:55<00:00,  2.10it/s]


The score can be evaluated as follows.

In [21]:
from cs589assignment1.utils.metrics import *
metrics = ["mrr", "ndcg@5", "ndcg@10"]

for component in components:
    print(component)
    score_dict = {(lang, metric): dict() for lang, metric in itertools.product(langs, metrics)}

    for lang, metric, algo in itertools.product(langs, metrics, algos):
        score_dict[(lang, metric)][algo] = compute_final_metric(lang, algo, component, metric)

    score_df = pd.DataFrame(score_dict)
    print(score_df)

title
         python                          java                     javascript  \
            mrr    ndcg@5   ndcg@10       mrr    ndcg@5   ndcg@10        mrr   
bm25   0.373923  0.390638  0.450905  0.367508  0.383647  0.441933   0.398748   
tfidf  0.386129  0.404605  0.461269  0.378340  0.395425  0.451339   0.405894   

                           
         ndcg@5   ndcg@10  
bm25   0.430057  0.482924  
tfidf  0.441018  0.492071  
question
         python                          java                     javascript  \
            mrr    ndcg@5   ndcg@10       mrr    ndcg@5   ndcg@10        mrr   
bm25   0.304285  0.315679  0.372764  0.310462  0.318669  0.376253   0.326257   
tfidf  0.258780  0.257418  0.322650  0.257047  0.253588  0.316805   0.272281   

                           
         ndcg@5   ndcg@10  
bm25   0.336612  0.398865  
tfidf  0.274559  0.336212  
answer
         python                          java                     javascript  \
            mrr    ndcg@5   ndcg

### 5. For BM25, by default k1=3, b=0.75. Revise the code to tune these parameters based on the MRR, NDCG@5 and NDCG@10 Write in your report with plot, e.g., using matplotlib. Analyze what are the best k1 and b? (20 points)

In [ ]:
from cs589assignment1.utils.metrics import *
k1_grid = [2.0, 2.5, 3.5]
b_grid  = [0.6, 0.8, 0.9]
langs = ["python"]
components = ["title", "question", "answer"]


records = []
for lang, component, k1, b in itertools.product(langs, components, k1_grid, b_grid):
    print(f"Running bm25 on {lang} {component} with k1={k1}, b={b}")
    run_retrieval_algorithm(lang, "bm25", component, k1=k1, b=b)

    mrr    = compute_final_metric(lang, "bm25", component, "mrr")
    ndcg5  = compute_final_metric(lang, "bm25", component, "ndcg@5")
    ndcg10 = compute_final_metric(lang, "bm25", component, "ndcg@10")

    records.append({"lang": lang, "component": component, "k1": k1, "b": b,
                    "mrr": mrr, "ndcg5": ndcg5, "ndcg10": ndcg10})

tune_df = pd.DataFrame(records)
display(tune_df.head())


Running bm25 on python title with k1=2.0, b=0.6


100%|██████████| 1000/1000 [06:08<00:00,  2.72it/s]


Running bm25 on python title with k1=2.0, b=0.8


100%|██████████| 1000/1000 [05:59<00:00,  2.78it/s]


Running bm25 on python title with k1=2.0, b=0.9


...

Running bm25 on python answer with k1=3.5, b=0.8


100%|██████████| 1000/1000 [05:13<00:00,  3.19it/s]


Running bm25 on python answer with k1=3.5, b=0.9


100%|██████████| 1000/1000 [05:32<00:00,  3.01it/s]


,lang,component,k1,b,mrr,ndcg5,ndcg10
0,python,title,2.0,0.6,0.405245,0.421774,0.481714
1,python,title,2.0,0.8,0.405245,0.421774,0.481714
2,python,title,2.0,0.9,0.405245,0.421774,0.481714
3,python,title,2.5,0.6,0.405245,0.421774,0.481714
4,python,title,2.5,0.8,0.405245,0.421774,0.481714


In [46]:
def best_by(df, component, metric):
    sub = df[df.component == component]
    row = sub.loc[sub[metric].idxmax()]
    return float(row.k1), float(row.b), float(row[metric])

for comp in components:
    for metric in ["mrr","ndcg5","ndcg10"]:
        k1,b,val = best_by(tune_df, comp, metric)
        print(f"{comp} best {metric}: k1={k1}, b={b}, score={val:.4f}")



title best mrr: k1=2.0, b=0.6, score=0.4052
title best ndcg5: k1=2.0, b=0.6, score=0.4218
title best ndcg10: k1=2.0, b=0.6, score=0.4817
question best mrr: k1=2.0, b=0.6, score=0.3043
question best ndcg5: k1=2.0, b=0.6, score=0.3157
question best ndcg10: k1=2.0, b=0.6, score=0.3728
answer best mrr: k1=2.0, b=0.6, score=0.2661
answer best ndcg5: k1=2.0, b=0.6, score=0.2695
answer best ndcg10: k1=2.0, b=0.6, score=0.3296


### Preparing Submission
You need to submit:

1. This notebook
2. Your report analysis for question 1-5